# B2.7 · Self-improving scaffolds

**Function B — Application Security with an AI SDLC → The Harnesses that Test CyberTravels**  ·  *Security of AI*

Builds on **[B2.6 · Failure taxonomy](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**.

| | |
|---|---|
| Tools used | Python, Docker, Kimi K2, Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A harness that rewrites its own prompts is a system that changes without a pull request. The improvement is real; so is the fact that nobody reviewed it, and that last week's evidence no longer describes what is running.

> **At CyberTravels.** A harness that rewrites its own prompts is a change to CyberTravels' security tooling with no pull request, no reviewer and no record.

## 2 · The framework

```
   run history --> proposed change --> prompt / tool / routing
                          |
                   +------v-------+
                   |  guardrail   |  eval suite must not regress
                   |  human diff  |  the change is a reviewable artefact
                   +--------------+

   a harness that rewrites itself is a system that changes with no pull request
```

A self-improving scaffold edits its own prompts, tools or routing based on how
well it is doing. It is genuinely effective, and it makes evaluation
non-optional rather than good practice.

The reason is mechanical. Optimisation moves toward whatever the metric rewards.
If the scaffold's metric is its own verifier, the scaffold will converge on
**satisfying the verifier**, which is only the same thing as doing the job if
the verifier is perfect. B2.2 established that yours is not.

So the loop is:

1. scaffold changes itself,
2. its own metric improves,
3. actual capability does not,
4. the dashboard is monotone and green.

The control is a **held-out signal**: a set of cases the scaffold cannot see,
cannot train on, and cannot reach — including through its logs. And the hard
part is not building it. It is keeping it held out.

## 3 · The model backend, and the change it proposes to itself

In [ ]:
# --- model backend: replay by default, real model when you configure one ----
# Nothing here is Anthropic- or vendor-specific beyond one URL and one header
# shape. Standard library only, so the notebook stays self-contained.
import json, os, urllib.error, urllib.request

# The cheapest current model on each side, which is what a lesson needs.
FRONTIER_DEFAULT   = "claude-haiku-4-5-20251001"
OPEN_WEIGHT_DEFAULT = "glm-4.6"
TIMEOUT = 60

def _kaggle_secret(name):
    """On Kaggle, a key lives in Add-ons -> Secrets rather than the environment.

    kaggle_secrets is pre-installed in the Kaggle image and absent everywhere
    else, so the import is guarded and the notebook needs no dependency. It also
    requires the notebook to have internet enabled, which on Kaggle requires a
    phone-verified account - see the note printed below.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("ANTHROPIC_API_KEY") or _kaggle_secret("ANTHROPIC_API_KEY"):
        os.environ.setdefault("ANTHROPIC_API_KEY",
                              os.environ.get("ANTHROPIC_API_KEY")
                              or _kaggle_secret("ANTHROPIC_API_KEY") or "")
        return "frontier", os.environ.get("MODEL", FRONTIER_DEFAULT)
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _anthropic(prompt, system, model, max_tokens, temperature):
    body = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    out = _post("https://api.anthropic.com/v1/messages", body,
                {"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                 "anthropic-version": "2023-06-01"})
    return "".join(b.get("text", "") for b in out.get("content", [])).strip()

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        fn = _anthropic if kind == "frontier" else _openai_compatible
        return fn(prompt, system, model, max_tokens, temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        detail = getattr(e, "code", None) or type(e).__name__
        print(f"   !! {kind} backend ({model}) failed: {detail} - using the replay,")
        print("      which is a replay and is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, set one of:")
    print()
    print("   frontier     export ANTHROPIC_API_KEY=...   # cheapest: " + FRONTIER_DEFAULT)
    print("   open weight  export OPENAI_BASE_URL=http://localhost:11434/v1 \\")
    print("                       OPENAI_API_KEY=ollama MODEL=glm-4.6")
    print()
    print("   On Kaggle: Add-ons -> Secrets, add ANTHROPIC_API_KEY, and switch")
    print("   Internet on in the notebook settings. Internet requires a")
    print("   phone-verified Kaggle account; without it DNS fails in the kernel")
    print("   and this lesson correctly stays on the replay.")

## 4 · The same lesson, against a real model

Everything below this point runs identically on three backends. Offline it uses
a deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `ANTHROPIC_API_KEY` set it calls a frontier
model; with `OPENAI_BASE_URL` set it calls any OpenAI-compatible endpoint,
which covers Ollama, vLLM and the hosted open-weight providers.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'This harness reports success when the test suite is green but the patch did not change behaviour. Propose one change to its verification step. One sentence.'

REPLAY = 'Require the test suite to fail on the pre-patch build and pass on the post-patch build, so a no-op patch cannot be reported as a fix.'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You improve agent harnesses. One sentence, no preamble.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("mentions the pre-fix build", any(w in answer.lower() for w in ("before", "pre-", "old build", "fail")))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, three possible backends. Offline the answer")
print("is the replay and is labelled as one; with a key it is the model's.")

## 5 · Demo — a scaffold optimising against its own judge

In [ ]:
import random

TASKS = [{"id": i, "input": i, "correct": i * 2} for i in range(1, 21)]

class Scaffold:
    """Adjusts its own strategy based on feedback. DETERMINISTIC — not a model."""
    def __init__(self):
        self.strategy = "double"        # starts correct
        self.history = []

    def solve(self, task):
        if self.strategy == "double":     return task["input"] * 2
        if self.strategy == "constant":   return 42
        return task["input"]

    def improve(self, score):
        """Move toward whatever scored better last round."""
        self.history.append((self.strategy, score))
        if len(self.history) >= 2 and self.history[-1][1] <= self.history[-2][1]:
            order = ["double", "constant", "identity"]
            self.strategy = order[(order.index(self.strategy) + 1) % 3]

def own_judge(task, answer):
    """The scaffold's own verifier: is the answer a plausible-looking number?"""
    return isinstance(answer, int) and answer > 0

def held_out_oracle(task, answer):
    """The truth. The scaffold never sees this."""
    return answer == task["correct"]

def evaluate(scaffold, verifier, tasks):
    return sum(verifier(t, scaffold.solve(t)) for t in tasks) / len(tasks)

s = Scaffold()
print(f"{'round':6s}{'strategy':11s}{'own judge':11s}{'held-out truth':15s}")
print("-" * 46)
for r in range(1, 7):
    own = evaluate(s, own_judge, TASKS)
    truth = evaluate(s, held_out_oracle, TASKS)
    print(f"{r:<6}{s.strategy:11s}{own:<11.2f}{truth:<15.2f}")
    s.improve(own)

## 6 · Where it breaks — the self-metric cannot fall

The judge scores 1.00 for every strategy, including `constant`, which returns 42 for everything. There is no gradient for the scaffold to follow toward correctness, and no signal that anything is wrong. The dashboard is green in every round.

Now let the scaffold drift, and watch the two curves separate.

In [ ]:
s2 = Scaffold()
s2.strategy = "constant"            # a drift that the judge cannot penalise
print(f"{'round':6s}{'strategy':11s}{'own judge':11s}{'held-out truth':15s}verdict")
print("-" * 62)
for r in range(1, 5):
    own = evaluate(s2, own_judge, TASKS)
    truth = evaluate(s2, held_out_oracle, TASKS)
    verdict = "looks fine" if own > 0.9 else "alert"
    real = "BROKEN" if truth < 0.9 else "ok"
    print(f"{r:<6}{s2.strategy:11s}{own:<11.2f}{truth:<15.2f}{verdict} / actually {real}")
    s2.improve(own)

## 7 · The control — a held-out set, and keeping it held out

Building the set is easy. The three ways it leaks are not obvious, and each one silently converts your held-out signal into another self-metric.

In [ ]:
LEAKS = [
 ("the scaffold reads its own logs, which contain held-out failures",
  "it optimises against them without anyone deciding to"),
 ("held-out cases are added to the prompt as 'examples of hard tasks'",
  "the most common leak, and it looks like good engineering"),
 ("the same corpus is used for both tuning and evaluation",
  "the number becomes a training metric and rises forever"),
 ("a human adjusts the scaffold after reading held-out results",
  "slower leak, same effect — the human is the gradient"),
]
for how, why in LEAKS:
    print(f"✗ {how}\n    → {why}\n")

def evaluation_is_sound(scaffold_can_read_logs, cases_in_prompt,
                        same_corpus, human_tunes_on_results):
    problems = []
    if scaffold_can_read_logs:    problems.append("scaffold can read held-out outcomes")
    if cases_in_prompt:           problems.append("held-out cases appear in the prompt")
    if same_corpus:               problems.append("tuning and eval share a corpus")
    if human_tunes_on_results:    problems.append("human closes the loop manually")
    return (not problems), problems

for label, args in (("as usually built", (True, True, True, True)),
                    ("after the fix",    (False, False, False, False))):
    ok, problems = evaluation_is_sound(*args)
    print(f"{label:18s} sound={ok}")
    for p in problems: print(f"      ⚠ {p}")

In [ ]:
# Verify: gate the scaffold on the held-out signal, not its own.
def gated_improve(scaffold, tasks, holdout_tasks):
    before = evaluate(scaffold, held_out_oracle, holdout_tasks)
    candidate = Scaffold(); candidate.strategy = "constant"
    after = evaluate(candidate, held_out_oracle, holdout_tasks)
    if after < before:
        return scaffold, f"REJECTED change: held-out {before:.2f} → {after:.2f}"
    return candidate, f"accepted: held-out {before:.2f} → {after:.2f}"

HOLDOUT = [{"id": 100+i, "input": 100+i, "correct": (100+i)*2} for i in range(10)]
good = Scaffold()
kept, why = gated_improve(good, TASKS, HOLDOUT)
print(why)
print("final strategy:", kept.strategy)
assert kept.strategy == "double"
print("\nThe scaffold's own judge would have accepted the change. The held-out")
print("oracle rejected it, which is the only reason the system still works.")

## What you just proved

The self-judge scores 1.00 in every round regardless of strategy, while held-out truth is 1.00 only for `double`. With the scaffold drifted to `constant`, the judge still reports 1.00 and "looks fine" while held-out truth is 0.00 and the system is BROKEN. The leak checklist flags all four paths, and the held-out gate rejects the change the self-judge would have accepted.

## Your turn

For any self-tuning component you run, answer one question: can it see the outcomes of its evaluation, through any path including logs? If yes, you have a self-metric with extra steps.

---

**Next → [B2.8 · Idempotency, replay and rollback](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*